# 🌾 Crop Yield & Weather Impact Analysis
**Mini Project – B.Tech IT (Second Year)**

**Aim:** To study how rainfall and temperature affect crop yield using graphs, and to make a simple yield prediction using Linear Regression.

**Dataset:** Crop Yield Prediction Dataset (Kaggle) – file name `yield_df.csv`

---
### Steps in this project
1. Load and understand the data
2. Clean the data
3. Make graphs (Exploratory Data Analysis)
4. Predict yield using Linear Regression
5. Conclusion

## Step 0: Import Libraries
These are the Python tools we use:
- **pandas** → to read and work with the data table
- **matplotlib / seaborn** → to make graphs
- **scikit-learn** → for the prediction model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
print("Libraries loaded successfully!")

## Step 1: Load the Data
**Before running this cell:**
1. Download the dataset from Kaggle (search: *"crop yield prediction dataset"*)
2. Extract the zip file and find **`yield_df.csv`**
3. Run the cell below and click **"Choose Files"** to upload `yield_df.csv`

In [ ]:
# Upload the file in Google Colab
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab - make sure yield_df.csv is in the same folder.")

df = pd.read_csv("yield_df.csv")
print("Data loaded!")

In [ ]:
# First 5 rows of the data
df.head()

In [ ]:
# Size of the data
print("Number of rows   :", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Column names and data types
df.info()

### What the columns mean
| Column | Meaning |
|---|---|
| Area | Country name |
| Item | Crop name (Rice, Wheat, Maize, etc.) |
| Year | Year of the record |
| hg/ha_yield | Crop yield in hectograms per hectare |
| average_rain_fall_mm_per_year | Yearly rainfall (mm) |
| pesticides_tonnes | Pesticides used (tonnes) |
| avg_temp | Average temperature (°C) |

## Step 2: Clean the Data
We will:
1. Remove the extra index column (`Unnamed: 0`)
2. Give the columns short, simple names
3. Convert yield from hg/ha to **kg/ha** (easier to understand; 1 hg = 0.1 kg)
4. Remove missing values and duplicate rows

In [ ]:
# 1. Remove the extra index column if it exists
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# 2. Rename columns
df = df.rename(columns={
    "Area": "Country",
    "Item": "Crop",
    "hg/ha_yield": "Yield",
    "average_rain_fall_mm_per_year": "Rainfall",
    "pesticides_tonnes": "Pesticides",
    "avg_temp": "Temperature"
})

# 3. Convert yield to kg per hectare
df["Yield"] = df["Yield"] / 10

# 4. Remove missing values and duplicates
print("Missing values before cleaning:")
print(df.isnull().sum())
before = len(df)
df = df.dropna().drop_duplicates()
print("\nRows removed:", before - len(df))
print("Rows remaining:", len(df))

df.head()

In [ ]:
# Basic statistics (mean, min, max, etc.)
df.describe().round(2)

In [ ]:
# Which crops are in the data?
print("Crops in dataset:")
print(df["Crop"].value_counts())

## Step 3: Graphs (Exploratory Data Analysis)
This is the most important part of the project. After each graph, write what you observe in the **Observation** box.

### Graph 1: Average Yield Over the Years

In [ ]:
yearly = df.groupby("Year")["Yield"].mean()

plt.plot(yearly.index, yearly.values, marker="o", color="green")
plt.title("Average Crop Yield Over the Years")
plt.xlabel("Year")
plt.ylabel("Average Yield (kg/ha)")
plt.show()

**Observation:** _Write here – e.g. Is the yield increasing or decreasing over time?_

### Graph 2: Average Yield of Each Crop

In [ ]:
crop_yield = df.groupby("Crop")["Yield"].mean().sort_values()

crop_yield.plot(kind="barh", color="orange")
plt.title("Average Yield of Each Crop")
plt.xlabel("Average Yield (kg/ha)")
plt.ylabel("Crop")
plt.show()

**Observation:** _Write here – e.g. Which crop gives the highest yield and which gives the lowest?_

### Graph 3: Rainfall vs Yield

In [ ]:
sns.scatterplot(data=df, x="Rainfall", y="Yield", alpha=0.3, color="blue")
plt.title("Rainfall vs Crop Yield")
plt.xlabel("Rainfall (mm per year)")
plt.ylabel("Yield (kg/ha)")
plt.show()

**Observation:** _Write here – e.g. Does more rainfall always give more yield?_

### Graph 4: Temperature vs Yield

In [ ]:
sns.scatterplot(data=df, x="Temperature", y="Yield", alpha=0.3, color="red")
plt.title("Temperature vs Crop Yield")
plt.xlabel("Average Temperature (°C)")
plt.ylabel("Yield (kg/ha)")
plt.show()

**Observation:** _Write here – e.g. At what temperatures is the yield highest?_

### Graph 5: Rainfall vs Yield for One Crop (Rice)
Different crops behave differently, so let's look at just one crop. You can change `"Rice, paddy"` to any crop name from the list above.

In [ ]:
one_crop = df[df["Crop"] == "Rice, paddy"]

sns.regplot(data=one_crop, x="Rainfall", y="Yield",
            scatter_kws={"alpha": 0.3}, line_kws={"color": "red"})
plt.title("Rainfall vs Yield for Rice")
plt.xlabel("Rainfall (mm per year)")
plt.ylabel("Yield (kg/ha)")
plt.show()

**Observation:** _Write here – e.g. Does the red trend line go up or down for rice?_

### Graph 6: Correlation Heatmap
Correlation shows how strongly two things are related:
- Close to **+1** → both increase together
- Close to **−1** → one increases, the other decreases
- Close to **0** → no clear relation

In [ ]:
corr = df[["Yield", "Rainfall", "Temperature", "Pesticides"]].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Weather Factors and Yield")
plt.show()

**Observation:** _Write here – e.g. Which factor has the strongest relation with yield?_

## Step 4: Predict Yield using Linear Regression
**Linear Regression** draws the best straight-line relationship between the inputs and the output.

- **Inputs (X):** Rainfall, Temperature
- **Output (y):** Yield

We use 80% of the data to **train** the model and 20% to **test** it.

In [ ]:
# Model 1: using only weather
X = df[["Rainfall", "Temperature"]]
y = df["Yield"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)

print("Model 1 (Weather only)")
print("R² Score :", round(r2_score(y_test, pred1), 3))
print("MAE      :", round(mean_absolute_error(y_test, pred1), 2), "kg/ha")

**How to read the results:**
- **R² Score** → between 0 and 1. Closer to 1 = better prediction.
- **MAE (Mean Absolute Error)** → on average, how far our prediction is from the real value.

The R² score here will be **low**. Why? Because yield depends a lot on **which crop** it is (potatoes give far more kg/ha than wheat). Weather alone can't explain that. So let's add the crop type.

In [ ]:
# Model 2: weather + crop type
# get_dummies converts crop names into 0/1 columns so the model can use them
X2 = pd.get_dummies(df[["Rainfall", "Temperature", "Crop"]], columns=["Crop"])
y2 = df["Yield"]

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)
pred2 = model2.predict(X2_test)

print("Model 2 (Weather + Crop type)")
print("R² Score :", round(r2_score(y2_test, pred2), 3))
print("MAE      :", round(mean_absolute_error(y2_test, pred2), 2), "kg/ha")

### Comparing the Two Models

In [ ]:
results = pd.DataFrame({
    "Model": ["Weather only", "Weather + Crop"],
    "R² Score": [round(r2_score(y_test, pred1), 3), round(r2_score(y2_test, pred2), 3)],
    "MAE (kg/ha)": [round(mean_absolute_error(y_test, pred1), 2),
                    round(mean_absolute_error(y2_test, pred2), 2)]
})
results

In [ ]:
# Actual vs Predicted graph for Model 2
plt.scatter(y2_test, pred2, alpha=0.3, color="purple")
plt.plot([y2_test.min(), y2_test.max()], [y2_test.min(), y2_test.max()], color="red")
plt.title("Actual vs Predicted Yield (Model 2)")
plt.xlabel("Actual Yield (kg/ha)")
plt.ylabel("Predicted Yield (kg/ha)")
plt.show()

**Observation:** _Points close to the red line = good predictions._

### Try Your Own Prediction
Change the values below and run the cell.

In [ ]:
crop_name   = "Rice, paddy"   # pick a crop from the list above
rainfall    = 1200            # mm per year
temperature = 26              # °C

new_data = pd.DataFrame(0, index=[0], columns=X2.columns)
new_data["Rainfall"] = rainfall
new_data["Temperature"] = temperature
new_data["Crop_" + crop_name] = 1

result = model2.predict(new_data)[0]
print(f"Crop        : {crop_name}")
print(f"Rainfall    : {rainfall} mm")
print(f"Temperature : {temperature} °C")
print(f"Predicted Yield: {result:.2f} kg/ha")

## Step 5: Conclusion
_Fill this in after you run all cells and look at your graphs._

1. The crop with the highest average yield is ______.
2. The yield over the years has ______ (increased / decreased).
3. Rainfall has a ______ (strong / weak) relation with yield.
4. Temperature has a ______ (strong / weak) relation with yield.
5. Using only weather, the R² score was ______. Adding crop type improved it to ______.
6. **Final conclusion:** Weather affects crop yield, but the type of crop matters even more. To predict yield well, we need both weather and crop information.

### Future Scope
- Add soil data (pH, nitrogen)
- Use India-specific state-wise data
- Try better models like Random Forest